# IATI project review, results and next steps

**Main review notebook — updated October 21, 2026.**

- Read this notebook for what changed, why it helps, results and file locations.
- The scope is **offline only**. Run All reads local evidence; it does not call model APIs, spend credits or write to Neo4j.
- The full code remains in the focused notebooks and Python modules linked below. This avoids copying implementations into a large notebook.
- The 26-test result is from the recorded review. An optional cell at the end can rerun offline checks.
- Meeting context below is based on the supplied summary; the linked document was not independently retrieved during this review.

## Quick summary

- Fixed **807 overwritten relationship records**; the corrected graph keeps **59,121 relationships**.
- Fixed false success scores for failed queries.
- Made the three offline notebooks reproducible and prepared public model-evaluation inputs.
- **No real-model comparison has run.** Fake responses test the workflow, not model quality.
- Next: prioritize confirmed free-tier access for **GLM**, with Kimi and Qwen as optional comparisons, agree on funding-data rules and validate a separate test database.
- **No API budget, credits or paid access are confirmed.**


## Current team status — October 21, 2026

- George confirmed that he has **not yet run the model APIs or the Neo4j load**. His work provides the implementation; completed validation is offline only.
- **GLM is the client's preferred candidate** for the first model pilot. The exact model/version, provider endpoint and access limits still need confirmation.
- Kimi and Qwen remain optional comparison candidates, not equal-priority commitments.
- Start with verified free-tier access. No API budget, credits or paid access are confirmed; do not assume Brent will cover charges.
- Memory and reranking remain proposed architecture work, not implemented or evaluated features.
- No live model results, model ranking or live database validation are claimed.


## Where everything is

Paths below are relative to the repository root. Clickable links are relative to this notebook.

| What | File or folder | What to do there |
| --- | --- | --- |
| This overview | `fa26/notebooks/00_review_and_next_steps.ipynb` | Read results, decisions and next steps |
| Baseline graph | [01 notebook](01_reproduce_summer_baseline_2026-10-15.ipynb) | Inspect data and the initial graph |
| Full graph/schema | [02 notebook](02_schema_draft_2026-10-15.ipynb) | Inspect preserved relationships and five diagnostic queries |
| Model diagnostics | [03 notebook](03_text2cypher_langgraph_eval_2026-10-15.ipynb) | Run synthetic controls; live mode is separate |
| Database import | [04 notebook](04_neo4j_aura_load_2026-10-15.ipynb) | Future isolated-database load; not part of Run All here |
| Reusable evaluation code | [text2cypher_eval.py](../src/text2cypher_eval.py) | Query guard, retries, scoring control flow, provider registry |
| Code labels | [iati_codelists.py](../src/iati_codelists.py) | Shared IATI labels for graph and loader |
| Offline runner | [run_offline_notebooks.py](../src/run_offline_notebooks.py) | Execute notebooks 01–03 |
| Evaluation tests | [test_text2cypher_eval.py](../tests/test_text2cypher_eval.py) | Regression checks for fixed bugs |
| Full 20-question benchmark | [benchmark/README.md](../../benchmark/README.md) | Question contracts, submissions and scoring |
| Public model inputs | [prepare_model_inputs.py](../../benchmark/prepare_model_inputs.py) | Export data/questions without answer keys |
| Original-branch audit | [review_george.py](../../analysis/review_george.py) | Reproduce the original issues from George's Git commit |
| Review evidence | [original findings](../../analysis/george_review_results.json), [offline validation](../../analysis/george_offline_validation.json) | Machine-readable saved results |
| Generated files | `fa26/data/interim/`, `fa26/data/processed/` | Local schema, logs and synthetic CSVs; ignored by Git |

Use the public-input exporter to create a new folder outside the evaluator checkout. Generated export folders are not committed.


### How to use this cell — open the saved review

- Open this notebook from the repository root or `fa26/notebooks` and select a Python 3 kernel.
- Select the code cell below and press **Shift+Enter**. Run this cell first; later cells use the files it loads.
- No values need changing. It finds the repository and reads the saved review results.
- The output shows the reviewed commit, validation date, live-run count and any source files changed since validation. A listed file means the saved check predates that edit; it does not automatically mean the code is broken.
- If the repository cannot be found, reopen the notebook from the project folder. If a results file is missing, restore it from Git.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "benchmark/manifest.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or fa26/notebooks")
original = json.loads((ROOT / "analysis/george_review_results.json").read_text())
validation = json.loads((ROOT / "analysis/george_offline_validation.json").read_text())
print("Repository located successfully.")
print("Original George commit:", original["george_commit"])
print("Recorded validation time (UTC):", validation.get("generated_at", "not recorded"))
print("Real model runs:", original["live_model_runs"])
changed = [name for name, digest in validation.get("source_sha256", {}).items()
           if not (ROOT / name).exists()
           or hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != digest]
print("Previously validated source files changed:", changed or "none")
print("This overview is new; the saved results describe the earlier code review.")


## Review provenance


Reviewed on October 20, 2026 (America/New_York). Scope: offline review.
George source: `e86605bf92f8c5aa98cbb165a3ecf8820b735df1`.
Integration base: Deniz's `c01c0c0b89960626e5ee3516cfb95d0484b2708c`.
The remote `george` branch was not modified. Changes are prepared on local `denizbranch`.


## Conclusion

George's work provides a useful graph/schema prototype and a five-question Text2Cypher harness. It does **not** contain evidence of a completed real-model comparison. George confirmed that the model API calls and Neo4j load have not been run. The `.env.example` lists configuration placeholders, not confirmed access or results.

The offline graph calculations reproduce after replacing the machine-specific input path at runtime. The reviewed integration now runs on the hash-verified 22-CSV benchmark snapshot, without a reference clone, API keys, `.env`, or Neo4j. Synthetic model controls exercise the harness only.


## Findings, evidence and repairs

| Finding | Reproduced evidence | Change / remaining limit |
| --- | --- | --- |
| Relationship records are silently overwritten | Original notebook 02 builds 58,314 edges; the source has 59,121 mapped relationship records. All 807 overwritten rows are in `rel_participates`; 402 organisation/activity pairs have multiple role values. | Use source-record edge identities in NetworkX and in the Neo4j loader. Both graphs retain all participation rows/roles. Duplicate source rows remain explicit; downstream questions must specify distinct-ID or row semantics. |
| A failed query can receive a correct score | An original execution failure followed by an empty gold result yields `match=True`. Two guard rejections can instead end in `KeyError` because `rows` was never initialized. | Failed generation, guard, execution and reference states are explicit. Failed queries never earn a match, including empty gold results. Regression tests cover both paths. |
| Regex guard overstates protection | Original regex permits `MATCH (n) CALL custom.proc() RETURN n`; only selected procedure names are blocked. | Reject all procedures, writes, comments, escaped identifiers and multiple statements conservatively. Require dedicated read-only credentials for future live evaluation, with read routing, timeouts and row caps. Regex is not authorization; server permissions must still be verified before live use. |
| Provider diagnostics can expose secrets | Notebook 03 prints a key prefix and environment values and writes raw errors/auth/proxy fields to a file. | Remove this diagnostics cell. Stored errors contain phase and exception class only. Registry metadata references credential variable names, not values. |
| Other-provider keys can use the wrong endpoint | Non-OpenAI providers allow an empty base URL, which leaves client defaults in control. | Require an explicit HTTPS base URL for those providers. Added registry tests. |
| Advertised dry run is not self-contained | Notebook 03 defaults to live mode and asserts a `.env` exists; notebooks 01/02/04 reference a local sibling checkout. | Default to synthetic mode, make `.env` optional, preserve injected environment values, use the pinned snapshot and an offline runner. |
| Timing omits work | Original generation overwrites previous-attempt timing; result timing excludes execution and comparison. | Record per-question harness wall time including retries, execution and comparison. It is not pure model inference latency. |
| Schema implementations drift | Notebook 02 has code mappings missing from notebook 04; the in-memory Activity lacks the advertised `hunger_basis`. | Share codelists and set `hunger_basis` consistently. Offline loader test checks row identity; no real database load was attempted. |
| Five diagnostics are not the full evaluation | George's Q1–Q5 do not share the meanings of benchmark Q01–Q20. The stub recognizes exact gold strings. | Keep diagnostic results separate. Provide a public-input exporter for the existing 20-question benchmark; exclude all answer keys, catalog, SQL baseline and prior results. No automatic conversion or score blending. |
| Financial questions need clearer contracts | Legacy Q3 drops negative/zero values, groups by names, and ranks numeric totals across currencies. Float sums are rounded to whole units. | Preserve this legacy calculation for reproducibility and explicitly label its positive-only contract in the model question. Do not present it as net funding or currency-comparable ranking. The 20-question benchmark uses signed Decimal arithmetic and explicit type/currency/period contracts where specified. Policy remains a team/stakeholder decision. |


## Executed results

- Original notebooks 01/02 reproduced from the exact George commit, with only setup paths supplied by the audit runner: 27,279 nodes; 31,484 / 58,314 edges; 55 / 0 isolated nodes.
- Reviewed notebook 01: 27,279 nodes and 32,291 edges before provider/receiver edges.
- Reviewed notebook 02: 27,279 nodes and **59,121** edges, with all 3,457 participation records and their roles preserved; no missing endpoints, duplicate node keys or blank node keys in the mapped tables.
- Reviewed notebook 03: five exact-gold synthetic controls matched; five invalid synthetic controls failed. **Zero live model runs**.
- Original data remains 505 activities and 23,423 transactions; no snapshot, answer-key or Gradio app changes.
- Existing benchmark regression suite: 11 tests passed. Reviewed Text2Cypher/loader suite: 13 tests passed. Public-input export suite: 2 tests passed.
- Dependency consistency check passed. Gradio app compatibility was rechecked after installing notebook dependencies.

Raw original-branch findings: `analysis/george_review_results.json`.
Reviewed run record with source hashes: `analysis/george_offline_validation.json`.
Local notebook logs, schema output and synthetic CSVs: `fa26/data/interim/` and `fa26/data/processed/` (ignored generated outputs).


### How to use this cell — read the recorded results

- Run the first code cell, then press **Shift+Enter** on the cell below.
- No values need changing. This displays the saved notebook results, node/relationship counts and the number of overwritten records found in the original code.
- This cell **does not rerun tests**. Use the optional final cell for a fresh check.
- If you see `NameError` for `validation` or `original`, rerun the first code cell.


In [ ]:
for item in validation["notebooks"]:
    print(item["notebook"], "—", item["status"])
    if "nodes" in item:
        print("  Nodes:", item["nodes"], "Relationships:", item["relationships"])
        print("  Participation roles preserved:", item.get("participation_roles_preserved"))
    if "synthetic_control_cases" in item:
        print("  Synthetic controls:", item["synthetic_control_cases"], "Live runs:", item["live_model_runs"])
print("Original overwritten participation rows:", original["participation_relationships"]["overwritten_rows"])
print("Organisation/activity pairs with different roles:", original["participation_relationships"]["endpoint_pairs_with_multiple_roles"])


## Reproduce

From the repository root, using Python 3.12:

```bash
python3 -m venv .venv
# Activate .venv using your shell before running the commands below.
python -m pip install -r fa26/spaces/requirements.txt -r fa26/requirements-george.txt
PYTHONDONTWRITEBYTECODE=1 python analysis/review_george.py
PYTHONDONTWRITEBYTECODE=1 python fa26/src/run_offline_notebooks.py
PYTHONDONTWRITEBYTECODE=1 python fa26/tests/test_text2cypher_eval.py
PYTHONDONTWRITEBYTECODE=1 python benchmark/test_benchmark.py
PYTHONDONTWRITEBYTECODE=1 python benchmark/test_model_inputs.py
```

The original-branch audit requires George's commit to be available locally; if absent, `git fetch origin george` retrieves it while it remains in that branch's history. The other checks use the integrated files and snapshot directly. The notebook runner executes code cells in order with a fresh namespace per notebook; it does not test browser widgets or Jupyter kernel registration. It rewrites ignored local logs on each successful run; the copied validation record in `analysis/` documents this review.

Prepare an evaluation bundle in a new destination outside `benchmark/`:

```bash
python benchmark/prepare_model_inputs.py ../iati-model-inputs
```

The destination must not already exist. The bundle contains only the 22 CSVs, manifest, public questions, blank response template and instructions. Hand only that bundle to the model/agent; do not give it the evaluator checkout.


## Meeting context: access, costs and proposed models

Updated October 21, 2026, using the supplied summary of the September 25 and October 9 meeting notes. Meeting context is based on the supplied team summary; no private meeting-document content or access link is included.

- Start with **free-tier hosting and models**. Actual access, quotas and model availability must be checked before a run; a proposed provider is not proof of free access.
- **GLM is now the client-preferred first candidate**, following the team update. Kimi and Qwen remain optional comparisons. The exact model/version and architecture remain open.
- George discussed LangGraph/API state, memory and reranking. The current five-question notebook implements query generation, guards, execution and comparison; it does not implement or validate a memory/reranking layer.
- Brent would explore funding, credits and support, but **no budget, credits or paid API access were confirmed**. Possible GLM/Z.AI support was discussed, not committed. Do not assume Brent agreed to pay API costs.
- Confirm free-tier access first. Any paid run needs separate explicit spending approval; no paid run is authorized by these notes. This review remains offline.
- API/hosting costs are separate from **humanitarian funding data**: funders, funding categories, promised versus actual amounts, dates and historical records. Preserve those distinctions and agree on calculation rules separately.


## Suggested next steps

1. Team review the relationship-identity correction and use distinct activity/organisation IDs when questions call for unique entities.
2. Agree on the financial contract: transaction type, currency, reporting period, treatment of corrections and whether ranking is per currency. Retain George's positive-only Q3 only as a labeled diagnostic.
3. Confirm free-tier GLM access and limits, then select its exact model ID. Consider Kimi and Qwen only after the first pilot. No budget or credits are confirmed; seek separate explicit approval only if paid access is needed. Use the same snapshot, prompt, schema, retrieval policy and retry budget for every model; preserve exact model IDs, raw responses, token usage and full latency.
4. Verify an isolated Neo4j load and read-only account before enabling real Text2Cypher. Run five diagnostics first; these do not replace the full benchmark.
5. Run the 20-question benchmark using the public bundle. Produce structured submissions with complete source evidence and abstain/clarify responses, then score with `benchmark/score.py`. Preserve original model responses and disclose manual adapters.
6. Develop a held-out question set before making general model-quality claims. The development answer key and diagnostics are already visible to the team.


## The 20 questions to use later

These are loaded from the existing public question file. Each contract says exactly what to count or sum. George's five diagnostic questions use different IDs and must not be treated as the first five of this benchmark. Reference answers are deliberately not displayed here. This review notebook still contains evaluator findings; give models only the exported public-input bundle.


### How to use this cell — see the evaluation questions

- Run the first code cell, then press **Shift+Enter** below.
- It prints all 20 public questions and the calculation rule for each. It does not call a model or display the answer key.
- No values need changing. To review or propose a question change, use `benchmark/questions.json` and the benchmark instructions; do not silently change a question while comparing existing runs.
- A count assertion failure means the question set no longer contains the expected 20 questions.


In [ ]:
questions = json.loads((ROOT / "benchmark/questions.json").read_text())
assert len(questions) == 20
for item in questions:
    print(f"{item['id']}: {item['question']}")
    print("  Rule:", item["contract"])
    print()


## Optional: rerun offline checks

Leave `RUN_OFFLINE_CHECKS = False` to read saved results only. Set it to `True` to run the three notebooks and the 26-test suite using this kernel's Python. Install `fa26/requirements-george.txt` into that environment first. This writes only local ignored notebook logs/results and temporary test files; it does not update the saved review record or call APIs. A failing command stops the cell.

The original-branch audit is separate because it requires George's exact Git commit locally. Its command is in the Reproduce section above. Notebook 04 and live model mode are never called by this cell.


### How to use this cell — optionally run fresh offline checks

- For a read-only walkthrough, leave `RUN_OFFLINE_CHECKS = False` and press **Shift+Enter**. The cell prints that checks were not rerun.
- To run fresh checks, first install `fa26/requirements-george.txt` into the Python environment used by this notebook's kernel. Then run the first code cell, change the flag below to `True`, and press **Shift+Enter**.
- The cell runs notebooks 01–03 and three test scripts. Expected results: three completed notebooks and **13 + 11 + 2 = 26 passing tests**.
- It writes local logs/results under `fa26/data/interim` and `fa26/data/processed`. It does not replace the saved review record in `analysis/`.
- If a command fails, read the printed error; the cell stops instead of claiming success. A missing-package error usually means the kernel uses a different environment from the one where dependencies were installed.
- Set the flag back to `False` before saving or sharing. This cell never runs notebook 04 or makes live model API calls.


In [ ]:
RUN_OFFLINE_CHECKS = False
if RUN_OFFLINE_CHECKS:
    import os
    import subprocess
    run_env = dict(os.environ, IATI_LIVE_EVAL="0", PYTHONDONTWRITEBYTECODE="1")
    scripts = ["fa26/src/run_offline_notebooks.py", "fa26/tests/test_text2cypher_eval.py",
               "benchmark/test_benchmark.py", "benchmark/test_model_inputs.py"]
    for script in scripts:
        result = subprocess.run([sys.executable, script], cwd=ROOT, env=run_env,
                                capture_output=True, text=True)
        print(script)
        print(result.stdout)
        print(result.stderr)
        result.check_returncode()
else:
    print("Checks not rerun. Displayed results are from the saved offline review.")
